In [3]:
from tensorflow.keras.utils import register_keras_serializable
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Layer

@register_keras_serializable()
class L2Normalization(Layer):
    def __init__(self, axis=1, **kwargs):
        super(L2Normalization, self).__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return K.l2_normalize(inputs, axis=self.axis)

@register_keras_serializable()
class EuclideanDistance(Layer):
    def __init__(self, **kwargs):
        super(EuclideanDistance, self).__init__(**kwargs)

    def call(self, inputs):
        x, y = inputs
        return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

In [6]:
from tensorflow.keras.models import load_model

siamese_model = load_model(
    "signature_siamese_model1.keras",
    custom_objects={
        "L2Normalization": L2Normalization,
        "EuclideanDistance": EuclideanDistance
    },
    compile=False
)

print(" Model loaded successfully")

 Model loaded successfully


In [7]:
import numpy as np
from tensorflow.keras.preprocessing import image

def load_image(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return img

def get_embedding(img_path):
    img = load_image(img_path)
    img = np.expand_dims(img, axis=0)
    # The embedding model is the second layer (the functional model inside Siamese)
    embedding_model = siamese_model.layers[2]  
    return embedding_model.predict(img)

In [9]:
import os

customer_database = {}

def register_customer(customer_id, folder_path):
    embeddings = []

    image_files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    if len(image_files) == 0:
        print("No images found in folder!")
        return

    for img_path in image_files:
        emb = get_embedding(img_path)
        embeddings.append(emb)

    customer_database[customer_id] = embeddings

    print(f"✅ Customer {customer_id} registered successfully.")
    print(f"Stored {len(embeddings)} signature samples.")

# Example:
register_customer("C101", "processed")  # your folder with signature images

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
✅ Customer C101 registered successfully.
Stored 12 signature samples.


In [11]:
def verify_signature(customer_id, test_signature_path, threshold):
    if customer_id not in customer_database:
        print("Customer not found!")
        return

    test_embedding = get_embedding(test_signature_path)
    stored_embeddings = customer_database[customer_id]

    distances = [np.linalg.norm(test_embedding - emb) for emb in stored_embeddings]
    avg_distance = np.mean(distances)

    print("Average Distance:", avg_distance)

    if avg_distance < threshold:
        confidence = (1 - avg_distance / threshold) * 100
        print("✅ Genuine Signature")
        print("Confidence:", round(confidence, 2), "%")
    else:
        confidence = (avg_distance / threshold - 1) * 100
        print("❌ Forged Signature")
        print("Confidence:", round(confidence, 2), "%")

# Example test
threshold = 0.66  # your previously computed threshold
verify_signature("C101", "genuine.jpeg", threshold)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Average Distance: 0.6018758
✅ Genuine Signature
Confidence: 8.81 %
